# 01. CMIP6 스타터: 캄보디아 건기 가뭄과 우기 홍수 지표 변화

**프로젝트:** Cambodia-Climate-Water-Energy. 2019년 건기 가뭄은 수력발전 감소와 전력 차질로 이어졌어요. 이 노트북은 그 출발점으로, 미래에 건기 강수와 가뭄 빈도가 어떻게 변하는지 봐요. 우기 홍수 지표도 함께 계산해요.

**로드맵 1단계 (1–3주차)** 용 노트북이에요. 이 노트북 하나로 다음을 해 봐요.

| 장 | 내용 | 주차 |
| --- | --- | --- |
| 0 | 환경 설정 | 1주 |
| 1 | 분석 설정 (영역, 기간, 시나리오) | 1주 |
| 2 | Pangeo CMIP6 카탈로그에서 모델 찾기 | 1주 |
| 3 | 캄보디아 영역 자료 불러오기 + 계절 사이클 | 1–2주 |
| 4 | 홍수 쪽 지표: Rx1day, Rx5day, R20mm (+ 가뭄 쪽 CDD) | 2주 |
| 5 | 가뭄 지표: SPI-3 직접 구현 | 3주 |
| 6 | 해석 질문과 다음 단계 (ECMWF 오픈데이터 맛보기) | 3주 |

- **실행 환경:** Google Colab 권장 (무료, 계정 불필요한 공개 자료만 사용)
- **자료:** Pangeo가 Google Cloud에 올린 CMIP6 zarr 자료 (`gs://cmip6`, 익명 접근)
- **주의:** 일자료(`day`)는 전 지구 단위 청크라서, 캄보디아만 잘라도 모델 해상도에 따라 기간당 1–3 GB를 내려받아요. 처음엔 기본 설정(모델 1개, 30년 2구간)으로 돌리세요.

## 0. 환경 설정
Colab에서는 아래 셀을 한 번 실행하세요. 로컬 conda 환경이라면 같은 패키지를 `conda install -c conda-forge`로 설치하면 돼요.

In [ ]:
# Colab: 처음 한 번만 실행 (1–2분)
!pip install -q xclim gcsfs zarr cartopy

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import gcsfs
import matplotlib.pyplot as plt
from scipy import stats

import xclim
from xclim import indices as xci
from xclim.core.units import convert_units_to

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False

print("xarray", xr.__version__, "| xclim", xclim.__version__, "| cartopy", HAS_CARTOPY)

## 1. 분석 설정
여기 값만 바꾸면 나머지가 따라와요. **연습 과제:** `FUTURE_EXP`를 `"ssp585"`로, `FUT`를 `("2071", "2100")`으로 바꿔 다시 돌려 보세요.

- 영역: 캄보디아를 감싸는 사각형 (경도 102.3–107.7°E, 위도 10.4–14.7°N). 해안·인접국 일부가 포함돼요. 국경 마스크는 2단계(6주)에서 `regionmask`로 해요.
- 기준기간 1985–2014는 CMIP6 historical 실험의 마지막 30년이에요.

In [ ]:
LON = (102.3, 107.7)
LAT = (10.4, 14.7)
MARGIN = 0.75            # 거친 격자에서도 격자점이 충분히 잡히도록 여유

BASE = ("1985", "2014")  # 기준기간 (historical)
FUT = ("2041", "2070")   # 미래기간 (21세기 중반)
FUTURE_EXP = "ssp245"    # "ssp126", "ssp245", "ssp370", "ssp585"

# 위에서부터 차례로 시도: 해상도가 거친 모델이 먼저 (다운로드가 작음)
PREFERRED_MODELS = ["MIROC6", "MPI-ESM1-2-LR", "IPSL-CM6A-LR", "ACCESS-CM2", "EC-Earth3"]

## 2. CMIP6 카탈로그에서 모델 찾기
Pangeo 카탈로그는 CSV 하나예요. 열(column)이 곧 CMIP6의 구조예요.

- `source_id` 모델 이름 · `experiment_id` 실험(historical, ssp245…) · `member_id` 앙상블 멤버(r1i1p1f1…)
- `table_id` 시간 해상도(`day` 일, `Amon` 월) · `variable_id` 변수(`pr` 강수) · `zstore` 실제 자료 위치

In [ ]:
cat = pd.read_csv("https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv")
print(f"카탈로그 항목 수: {len(cat):,}")

def find_store(model, exp, table, var="pr"):
    # 조건에 맞는 zarr 경로 하나를 고른다 (r1i1p1f1, gn 격자 우선)
    sub = cat[(cat.source_id == model) & (cat.experiment_id == exp)
              & (cat.table_id == table) & (cat.variable_id == var)]
    if sub.empty:
        return None
    sub = sub.assign(
        _m=(sub.member_id != "r1i1p1f1").astype(int),
        _g=(sub.grid_label != "gn").astype(int),
    ).sort_values(["_m", "_g", "member_id", "version"], ascending=[True, True, True, False])
    return sub.iloc[0]

NEEDED = [("historical", "day"), (FUTURE_EXP, "day"), ("historical", "Amon"), (FUTURE_EXP, "Amon")]

MODEL, STORES = None, {}
for m in PREFERRED_MODELS:
    rows = {k: find_store(m, *k) for k in NEEDED}
    if all(r is not None for r in rows.values()):
        MODEL, STORES = m, rows
        break

assert MODEL, "조건을 모두 만족하는 모델이 없어요. PREFERRED_MODELS를 늘려 보세요."
print("선택한 모델:", MODEL)
for (exp, tab), r in STORES.items():
    print(f"  {exp:10s} {tab:5s} {r.member_id:10s} {r.grid_label}  {r.zstore}")

In [ ]:
# 탐색: 이 시나리오에 일강수(day pr)가 있는 모델은 몇 개일까? (2단계 다중 모델 분석의 후보)
have = cat[(cat.experiment_id == FUTURE_EXP) & (cat.table_id == "day") & (cat.variable_id == "pr")]
print(len(have.source_id.unique()), "개 모델:", sorted(have.source_id.unique()))

## 3. 캄보디아 영역 자료 불러오기
`open_zarr`는 자료를 바로 내려받지 않아요(lazy). `.sel()`로 영역·기간을 자른 다음 `.load()`할 때 실제로 받아요.

단위 변환: CMIP6 강수는 `kg m-2 s-1`(= mm/s)이에요. 하루는 86,400초라서 mm/day로 바꾸면 86,400을 곱한 값이 돼요. xclim의 `convert_units_to`가 단위 속성을 보고 알아서 바꿔 줘요.

In [ ]:
gcs = gcsfs.GCSFileSystem(token="anon")

def open_pr(row, period):
    ds = xr.open_zarr(gcs.get_mapper(row.zstore), consolidated=True)
    ds = ds.rename({k: v for k, v in {"latitude": "lat", "longitude": "lon"}.items() if k in ds.dims})
    ds = ds.sortby("lat")
    lon0, lon1 = LON
    if float(ds.lon.min()) < 0:          # -180..180 격자 대비
        ds = ds.assign_coords(lon=((ds.lon + 360) % 360)).sortby("lon")
    pr = ds["pr"].sel(
        lat=slice(LAT[0] - MARGIN, LAT[1] + MARGIN),
        lon=slice(lon0 - MARGIN, lon1 + MARGIN),
        time=slice(*period),
    )
    return pr.load()

def area_mean(da):
    w = np.cos(np.deg2rad(da.lat))
    return da.weighted(w).mean(("lat", "lon"))

In [ ]:
%%time
# 월자료 (작아요): 계절 사이클과 SPI용. 미래는 2015년부터 연속으로 받아 SPI 누적에 끊김이 없게 해요.
prm_hist = open_pr(STORES[("historical", "Amon")], BASE)
prm_fut  = open_pr(STORES[(FUTURE_EXP, "Amon")], ("2015", FUT[1]))
print(prm_hist.sizes, "| 격자점:", prm_hist.lat.size, "x", prm_hist.lon.size)

In [ ]:
# 계절 사이클: 캄보디아 우기(대략 5–10월)가 모델에서 잘 보이나요?
def clim_mmday(pr):
    return area_mean(convert_units_to(pr, "mm/d")).groupby("time.month").mean()

c_h = clim_mmday(prm_hist)
c_f = clim_mmday(prm_fut.sel(time=slice(*FUT)))

fig, ax = plt.subplots(figsize=(8, 4))
months = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]
ax.plot(range(1, 13), c_h, "o-", label=f"historical {BASE[0]}–{BASE[1]}")
ax.plot(range(1, 13), c_f, "s-", label=f"{FUTURE_EXP} {FUT[0]}–{FUT[1]}")
ax.set_xticks(range(1, 13)); ax.set_xticklabels(months)
ax.set_ylabel("precipitation (mm/day)")
ax.set_title(f"Cambodia box mean monthly precipitation, {MODEL}")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

print("연강수 변화: %.0f → %.0f mm/yr (%+.1f%%)" % (
    c_h.mean() * 365, c_f.mean() * 365, (c_f.mean() / c_h.mean() - 1) * 100))

# 건기(1–4월)와 우기(5–10월)를 나눠 보기: 수력발전에는 건기 강수와 직전 우기 누적이 중요해요
dry, wet = [1, 2, 3, 4], [5, 6, 7, 8, 9, 10]
for name, ms in [("건기 1–4월", dry), ("우기 5–10월", wet)]:
    h, f = float(c_h.sel(month=ms).mean()), float(c_f.sel(month=ms).mean())
    print(f"{name}: {h:.2f} → {f:.2f} mm/day ({(f / h - 1) * 100:+.1f}%)")

> Colab의 기본 글꼴은 한글을 못 그려서 그림 라벨은 영어로 두었어요. 한글이 필요하면 `!apt -qq install fonts-nanum` 후 `plt.rc("font", family="NanumGothic")`을 쓰세요.

### 일자료 불러오기 (시간이 좀 걸려요)

In [ ]:
%%time
prd_hist = convert_units_to(open_pr(STORES[("historical", "day")], BASE), "mm/d")
prd_fut  = convert_units_to(open_pr(STORES[(FUTURE_EXP, "day")], FUT), "mm/d")
print("달력:", prd_hist.time.dt.calendar, "| 일수:", prd_hist.time.size, prd_fut.time.size)

# 다시 받지 않도록 저장해 두기 (Colab이면 Google Drive에 저장 권장)
prd_hist.to_netcdf(f"pr_day_{MODEL}_hist_cambodia.nc")
prd_fut.to_netcdf(f"pr_day_{MODEL}_{FUTURE_EXP}_cambodia.nc")

## 4. 극한 강수 지표 (xclim)
홍수와 직결되는 지표 3개와 가뭄 쪽 지표 1개를 계산해요. 모두 ETCCDI(기후변화 탐지 지수 전문가팀) 표준 지수예요.

| 지표 | 의미 | 리스크 연결 |
| --- | --- | --- |
| Rx1day | 연 최대 1일 강수량 | 돌발홍수, 도시 침수 |
| Rx5day | 연 최대 5일 누적 강수량 | 하천 홍수 (메콩·톤레삽 유역은 더 긴 누적이 중요) |
| R20mm | 일강수 20 mm 이상인 날 수 | 호우 빈도 |
| CDD | 최장 연속 무강수일 (일강수 1 mm 미만) | 건기 가뭄, 농업용수 |

CDD는 캄보디아 건기(11–4월)가 해를 넘기므로 **7월 시작 연도**(`YS-JUL`)로 계산해요.

In [ ]:
INDICES = {
    "Rx1day": lambda p: xci.max_1day_precipitation_amount(p, freq="YS"),
    "Rx5day": lambda p: xci.max_n_day_precipitation_amount(p, window=5, freq="YS"),
    "R20mm":  lambda p: xci.wetdays(p, thresh="20 mm/day", freq="YS"),
    "CDD":    lambda p: xci.maximum_consecutive_dry_days(p, thresh="1 mm/day", freq="YS-JUL"),
}

res = {}
for name, f in INDICES.items():
    h, fu = f(prd_hist), f(prd_fut)
    if name == "CDD":                    # 7월 시작 연도: 첫·마지막 불완전 연도 제외
        h, fu = h.isel(time=slice(1, -1)), fu.isel(time=slice(1, -1))
    res[name] = {"hist": h, "fut": fu}
    print(f"{name:7s} 단위={h.attrs.get('units', '')!s:8s} "
          f"기준 {float(area_mean(h).mean()):7.1f} → 미래 {float(area_mean(fu).mean()):7.1f}")

In [ ]:
# 지도: 기준기간 대비 변화. 강수량 지표는 %, 일수 지표는 일(day) 변화.
def change(name):
    h, fu = res[name]["hist"].mean("time"), res[name]["fut"].mean("time")
    if name in ("Rx1day", "Rx5day"):
        return (fu / h - 1) * 100, "%"
    return fu - h, "days"

proj = ccrs.PlateCarree() if HAS_CARTOPY else None
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2),
                         subplot_kw={"projection": proj} if HAS_CARTOPY else {})
for ax, name in zip(axes, INDICES):
    d, unit = change(name)
    lim = float(np.nanmax(np.abs(d))) or 1
    cmap = "BrBG_r" if name == "CDD" else "BrBG"     # 갈색 = 더 건조한 방향
    kw = dict(ax=ax, cmap=cmap, vmin=-lim, vmax=lim, cbar_kwargs={"label": unit, "shrink": 0.8})
    if HAS_CARTOPY:
        d.plot.pcolormesh(transform=ccrs.PlateCarree(), **kw)
        ax.coastlines(lw=0.6); ax.add_feature(cfeature.BORDERS, lw=0.6)
        ax.set_extent([LON[0] - 1, LON[1] + 1, LAT[0] - 1, LAT[1] + 1])
    else:
        d.plot.pcolormesh(**kw)
    ax.set_title(f"{name} change")
fig.suptitle(f"{MODEL} {FUTURE_EXP}: {FUT[0]}–{FUT[1]} vs {BASE[0]}–{BASE[1]}", y=1.03)
plt.show()

In [ ]:
# 영역 평균의 연도별 분포: 평균만이 아니라 '극단적인 해'가 늘어나는지 보기
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, name in zip(axes, INDICES):
    h = area_mean(res[name]["hist"]).values
    fu = area_mean(res[name]["fut"]).values
    ax.boxplot([h, fu], widths=0.5)
    ax.set_xticks([1, 2]); ax.set_xticklabels(["hist", FUTURE_EXP])
    ax.set_title(name); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**해석 포인트**
- 모델 격자가 1–2°라 캄보디아가 몇 개 격자로만 표현돼요. 지도는 공간 패턴보다 **변화의 방향과 크기**로 읽으세요.
- 영역 평균은 극한값을 희석해요. 격자별로 지수를 계산한 뒤 평균한 것(위 방식)과, 영역 평균 강수로 지수를 계산한 것은 다른 값이에요. 왜 그런지 메모해 두세요.

## 5. 가뭄 지표: SPI-3 직접 구현
SPI(표준강수지수)는 누적 강수량을 기준기간 분포에 맞춘 뒤 표준정규분포 값으로 바꾼 지수예요.

1. 월강수를 3개월 이동 합산해요 (SPI-3: 계절 규모 가뭄, 농업 가뭄과 관련).
2. 달(month)마다 따로, 기준기간 값으로 **감마분포**를 적합해요 (계절성 제거).
3. 강수 0인 경우의 비율 q를 섞은 누적확률 H(x) = q + (1 − q)·G(x)를 구해요.
4. H(x)를 표준정규분포의 분위수로 바꾸면 SPI예요. SPI ≤ −1은 중간 이상 가뭄, ≤ −1.5는 심한 가뭄이에요.

미래 값도 **기준기간 분포로** 변환하므로, 미래 SPI가 낮아지면 "지금 기준으로 더 건조해진다"는 뜻이에요. xclim에도 `standardized_precipitation_index`가 있어요. 직접 구현으로 원리를 익힌 뒤 xclim 결과와 비교해 보는 게 3주차 과제예요.

In [ ]:
def spi_1d(monthly_mm, months, window=3, cal_mask=None):
    # monthly_mm: 월 누적강수 (mm/month) 1차원 배열, months: 각 값의 달(1–12)
    # cal_mask: 분포 적합에 쓸 기준기간 위치 (True/False)
    acc = pd.Series(monthly_mm).rolling(window, min_periods=window).sum().to_numpy()
    out = np.full_like(acc, np.nan, dtype=float)
    for m in range(1, 13):
        sel = months == m
        ref = acc[sel & cal_mask]
        ref = ref[~np.isnan(ref)]
        q = np.mean(ref <= 0)                       # 무강수 비율
        a, loc, scale = stats.gamma.fit(ref[ref > 0], floc=0)
        x = acc[sel]
        H = q + (1 - q) * stats.gamma.cdf(np.where(x > 0, x, 0), a, loc=0, scale=scale)
        H = np.where(x <= 0, q, H)
        out[sel] = stats.norm.ppf(np.clip(H, 1e-6, 1 - 1e-6))
    return out

# 기준기간 + 2015년 이후를 이어 붙여 연속 시계열 만들기
prm = xr.concat([prm_hist, prm_fut], dim="time")
days = prm.time.dt.days_in_month
pr_mon_mm = area_mean(convert_units_to(prm, "mm/d")) * days     # mm/month

years = prm.time.dt.year.values
is_base = (years >= int(BASE[0])) & (years <= int(BASE[1]))
spi3 = xr.DataArray(
    spi_1d(pr_mon_mm.values, prm.time.dt.month.values, window=3, cal_mask=is_base),
    coords={"time": prm.time}, dims="time", name="SPI3",
)
print("기준기간 SPI-3 평균 %.2f, 표준편차 %.2f (0과 1에 가까워야 정상)" % (
    float(spi3[is_base].mean()), float(spi3[is_base].std())))

In [ ]:
yrs = years + (prm.time.dt.month.values - 0.5) / 12
v = spi3.values

fig, ax = plt.subplots(figsize=(14, 3.8))
ax.fill_between(yrs, 0, v, where=v >= 0, color="#2a7fb8", lw=0, step="mid")
ax.fill_between(yrs, 0, v, where=v < 0, color="#b8762a", lw=0, step="mid")
for th in (-1, -1.5):
    ax.axhline(th, color="k", lw=0.6, ls="--")
ax.axvspan(int(FUT[0]), int(FUT[1]) + 1, color="grey", alpha=0.12, label="future window")
ax.set_ylabel("SPI-3"); ax.set_title(f"Cambodia box SPI-3, {MODEL} historical + {FUTURE_EXP}")
ax.legend(loc="lower left"); plt.show()

def freq(mask, th):
    return float(np.nanmean(v[mask] <= th) * 100)

is_fut = (years >= int(FUT[0])) & (years <= int(FUT[1]))
for th, label in [(-1.0, "moderate+ (SPI<=-1)"), (-1.5, "severe+ (SPI<=-1.5)")]:
    print(f"{label:22s}: 기준 {freq(is_base, th):5.1f}%  →  미래 {freq(is_fut, th):5.1f}% of months")

## 6. 해석 질문과 다음 단계

**1단계 요약 메모를 쓸 때 답해 볼 질문 (3주차 산출물)**
1. 건기(1–4월) 강수는 늘었나요, 줄었나요? 2019년 같은 건기 가뭄은 미래에 더 잦아질까요?
2. 이 모델에서 우기 강수와 극한 강수(Rx5day)는 같은 방향으로 변하나요? 다르다면 홍수 리스크에 어떤 의미인가요?
3. 연강수가 늘어도 SPI 가뭄 빈도가 줄지 않을 수 있을까요? (계절 분포, 건기 길이를 생각해 보세요)
4. 모델 하나의 결과를 정책 문서에 바로 쓰면 안 되는 이유는? → 2단계(다중 모델, 관측 비교, 편의보정)로 이어져요.

**연습 과제**
- `FUTURE_EXP = "ssp585"`, `FUT = ("2071", "2100")`으로 다시 돌려 시나리오·시기 차이 비교하기
- SPI-6(`window=6`)도 계산해 보고 SPI-3과 비교하기
- `xclim.indices.standardized_precipitation_index`로 같은 결과가 나오는지 확인하기
- 극한 지표에 `R95pTOT`(상위 5% 호우가 연강수에서 차지하는 비율) 추가하기

### 다음 노트북: 02. CHIRPS로 본 2019년 가뭄 (4주차)
관측자료로 2019년 건기가 1981년 이후 몇 번째로 건조했는지 확인해요. 여기서 만든 `spi_1d` 함수를 그대로 다시 써요.

### 미리보기: 4단계 조기경보 (10주차)
아래는 ECMWF 오픈데이터에서 최신 앙상블 강수 예보를 받는 예시예요. 지금은 실행하지 않아도 돼요.

In [ ]:
# !pip install -q ecmwf-opendata cfgrib eccodes
# from ecmwf.opendata import Client
# client = Client(source="ecmwf")                 # source="aws" 등도 가능
# client.retrieve(type="pf", param="tp", step=[0, 168], target="ens_tp.grib2")   # IFS 앙상블 총강수
# ens = xr.open_dataset("ens_tp.grib2", engine="cfgrib")
# # 7일 누적강수 = step 168h 값 - step 0h 값 (m → mm는 ×1000)
# # 다음: 캄보디아 영역에서 '50 mm 초과 확률' = 앙상블 멤버 중 초과 비율